# Автоматизация batch-обработки видео с патчами

**Этап:** IV квартал 2025 — Валидация защит и финализация

**Источник:** `patch_video.ipynb` (ячейки 19+)

---

# for pathces

In [ ]:
!pip install ffmpeg-python opencv-python torch numpy

In [ ]:
import cv2
import torch
import random
import numpy as np
from tqdm.notebook import tqdm
import os

In [ ]:
class VideoProcessor:
    def __init__(self, patch_path, frames_mask: list):
        self.patch_path = patch_path
        self.frames_mask = frames_mask
        self.patch = self.load_patch(self.patch_path)

    def load_patch(self, patch_path):
        # Load and preprocess the patch
        patch = torch.load(patch_path).permute(1, 2, 0)  # Assuming patch is stored as torch tensor
        patch = self.to_rgb(self.sigmoid(patch)).numpy()

        return patch

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def to_rgb(x):
        return (x * 255).byte()

    def apply_patch_to_frame(self, frame):
        # Randomly choose a position to place the patch
        patch_x = random.randint(0, frame.shape[1] - self.patch.shape[1])
        patch_y = random.randint(0, frame.shape[0] - self.patch.shape[0])

        # Place the flipped patch on the frame
        frame[patch_y:patch_y + self.patch.shape[0], patch_x:patch_x + self.patch.shape[1]] = np.flip(self.patch, axis=2)
        return frame

    def update_video_paths(self, input_video_path, output_video_path):
        self.input_video_path = input_video_path
        self.output_video_path = output_video_path

    def process_video(self):
        # Load video
        cap = cv2.VideoCapture(self.input_video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {self.input_video_path}")

        # Get video properties
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Set up output video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(self.output_video_path, fourcc, fps, (frame_width, frame_height))

        # Process frames
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Apply patch to every Nth frame
            if self.frames_mask[frame_idx]:
                frame = self.apply_patch_to_frame(frame)

            # Write the frame to the output video
            out.write(frame)
            frame_idx += 1

        # Release resources
        cap.release()
        out.release()


def process_multiple_videos(processor, video_paths):
    for idx, (input_video, output_video) in tqdm(enumerate(video_paths)):
        processor.update_video_paths(input_video, output_video)
        processor.process_video()

# for one vid

In [ ]:
# Example usage
patch_file = '/content/25_64_epoch_4_ViT-SigLIP_so_patch_imgs_10203.pt'
frames_mask = [
    # [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
    [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
    [1., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
    [1., 0., 0., 0., 0., 1., 1., 0., 0., 0.],
    [1., 0., 0., 0., 0., 1., 1., 0., 1., 0.],
    [1., 0., 0., 0., 0., 1., 1., 1., 1., 0.],
    [1., 0., 0., 1., 1., 1., 0., 1., 1., 0.],
    [1., 0., 0., 1., 1., 1., 1., 1., 1., 0.],
    [1., 0., 0., 1., 1., 1., 1., 1., 1., 1.],
    [1., 0., 1., 1., 1., 1., 1., 1., 1., 1.],
    # [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]
    ]

processor = VideoProcessor(patch_file, frames_mask)

name = 'dzen'
save_dir_path = f'/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/{name} vids/car crash/{name} mp4 vids_{sum(frames_mask)}/'
os.makedirs(save_dir_path, exist_ok=True)

data_dir_path = f'/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/{name} mp4 vids/'

<ipython-input-89-9006be8d0088>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  patch = torch.load(patch_path).permute(1, 2, 0)  # Assuming patch is stored as torch tensor


In [ ]:
# List of tuples containing input/output video paths
video_paths = []
for vid_name in os.listdir(data_dir_path):
    if vid_name.endswith('.mp4'):
        video_paths.append(
            (
                os.path.join(data_dir_path, vid_name),
                os.path.join(save_dir_path, vid_name)
                )
        )


In [ ]:
# !rm -rf "/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/dzen mp4 vids_1"
# !rm -rf "/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/tiktok mp4 vids_1"

In [ ]:
# Process all 200 videos
process_multiple_videos(processor, video_paths)


0it [00:00, ?it/s]

In [ ]:
len(os.listdir("/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/tiktok mp4 vids_1"))

200

# for all vids mask

In [ ]:
import cv2
import torch
import random
import numpy as np
from tqdm.notebook import tqdm
import os

In [ ]:
# Example usage
patch_file = '/content/0_64_asshole_epoch_5_ViT-SigLIP_so_patch_imgs_10203.pt'
frames_mask = [
    # [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
    [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
    [1., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
    [1., 0., 0., 0., 0., 0., 0., 1., 1., 0.],
    [0., 0., 0., 0., 0., 1., 1., 1., 0., 1.],
    [0., 0., 0., 0., 0., 1., 1., 1., 1., 1.],
    [0., 0., 0., 0., 1., 1., 1., 1., 1., 1.],
    [0., 1., 0., 0., 1., 1., 1., 1., 1., 1.],
    [0., 1., 1., 0., 1., 1., 1., 1., 1., 1.],
    [1., 1., 1., 0., 1., 1., 1., 1., 1., 1.],
 [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]
    ]

name = 'dzen'
data_dir_path = f'/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/{name} mp4 vids/'

for fr_mask in tqdm(frames_mask):
    processor = VideoProcessor(patch_file, fr_mask)
    # save_dir_path = f'/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/tiktok vids/car crash'
    save_dir_path = f'/content/drive/MyDrive/Colab_Notebooks/CV/papers/data/HSE_Project_attacks/data/{name} vids/ass/{name} mp4 vids_{sum(fr_mask)}/'
    os.makedirs(save_dir_path, exist_ok=True)


    # List of tuples containing input/output video paths
    video_paths = []
    for vid_name in os.listdir(data_dir_path):
        if vid_name.endswith('.mp4'):
            video_paths.append(
                (
                    os.path.join(data_dir_path, vid_name),
                    os.path.join(save_dir_path, vid_name)
                    )
            )


    # Process all 200 videos
    process_multiple_videos(processor, video_paths)



  0%|          | 0/10 [00:00<?, ?it/s]

<ipython-input-3-e7279d784c4b>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  patch = torch.load(patch_path).permute(1, 2, 0)  # Assuming patch is stored as torch tensor


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]